In [644]:
import numpy

In [646]:
def Niddleman_Wunsch(seq1, seq2, scores):
    '''
    seq1, seq2 - последовательности
    scores - бонусы и штрафы в порядке: (match_bonus, mismatch_penalty, gap_penalty)
    '''
    match_bonus, mismatch_penalty, gap_penalty = scores

    path_diag, path_up, path_left = 0, 1, -1   # 0 - диагональ, 1 - сверху, -1 - слева

    N, M = len(seq1), len(seq2)
    best_score = np.zeros((M + 1, N + 1))   # матрица скоров
    path = np.zeros((M + 1, N + 1))         # матрица направлений

    # Инициализация
    best_score[0, 0] = 0
    path[0, 0] = 0

    # Первая строка (seq1)
    for i in range(1, N + 1):
        best_score[0, i] = best_score[0, i-1] + gap_penalty
        path[0, i] = path_left

    # Первый столбец (seq2)
    for j in range(1, M + 1):
        best_score[j, 0] = best_score[j-1, 0] + gap_penalty
        path[j, 0] = path_up

    # Прямой ход
    for i in range(1, N + 1):
        for j in range(1, M + 1):
            if seq1[i-1] == seq2[j-1]:                           # мэтч
                diag = best_score[j-1, i-1] + match_bonus
            else:                                                # мисмэтч
                diag = best_score[j-1, i-1] + mismatch_penalty
                
            up = best_score[j-1, i] + gap_penalty                # гэп сверху

            left = best_score[j, i-1] + gap_penalty              # гэп слева

            best_sc = max(diag, up, left)
            best_score[j, i] = best_sc

            if best_sc == diag:
                path[j, i] = path_diag
            elif best_sc == up:
                path[j, i] = path_up
            else:
                path[j, i] = path_left

    # Обратный ход
    seq1_align, seq2_align = "", ""
    pos_1, pos_2 = N, M
    while pos_1 != 0 or pos_2 != 0:
        came_from = path[pos_2, pos_1]
        if came_from == 0:
            seq1_align = seq1[pos_1-1] + seq1_align
            seq2_align = seq2[pos_2-1] + seq2_align
            pos_1 -= 1
            pos_2 -= 1
        elif came_from == -1:
            seq1_align = seq1[pos_1-1] + seq1_align
            seq2_align = '-' + seq2_align
            pos_1 -= 1
        elif came_from == 1:
            seq1_align = '-' + seq1_align
            seq2_align = seq2[pos_2-1] + seq2_align
            pos_2 -= 1

    return seq1_align, seq2_align, best_score[M, N], best_score

In [648]:
def Needleman_Wunsch_Affine(seq1, seq2, scores):
    """
    seq1, seq2 - последовательности
    scores - бонусы и штрафы в порядке: (match_bonus, mismatch_penalty, gap_open, gap_extend)
    """
    match_bonus, mismatch_penalty, gap_open, gap_extend = scores
    
    n, m = len(seq1), len(seq2)
    
    # Матрицы для очков
    M = np.full((n+1, m+1), -float('inf'))   # состояние замены
    Ix = np.full((n+1, m+1), -float('inf'))  # гэп в seq1
    Iy = np.full((n+1, m+1), -float('inf'))  # гэп в seq2
    
    # 0 - из M, 1 - из Ix, 2 - из Iy
    trace_M = np.zeros((n+1, m+1), dtype=int)
    trace_Ix = np.zeros((n+1, m+1), dtype=int)
    trace_Iy = np.zeros((n+1, m+1), dtype=int)
    
    # Инициализация
    M[0, 0] = 0
    # Для Ix: первый столбец (j=0)
    for i in range(1, n+1):
        Ix[i, 0] = -(gap_open + gap_extend * (i-1))
        trace_Ix[i, 0] = 1
    # Для Iy: первая строка (i=0)
    for j in range(1, m+1):
        Iy[0, j] = -(gap_open + gap_extend * (j-1))
        trace_Iy[0, j] = 2
    
    # Прямой ход
    for i in range(1, n+1):
        for j in range(1, m+1):
            s = match_bonus if seq1[i-1] == seq2[j-1] else mismatch_penalty
            
            # Состояние M[i,j]
            candidates = [M[i-1, j-1], Ix[i-1, j-1], Iy[i-1, j-1]]
            best_val = max(candidates)
            best_idx = candidates.index(best_val)  # 0,1,2
            M[i, j] = best_val + s
            trace_M[i, j] = best_idx
            
            # Состояние Ix[i,j]
            open_gap = M[i, j-1] - gap_open
            extend = Ix[i, j-1] - gap_extend
            if open_gap >= extend:
                Ix[i, j] = open_gap
                trace_Ix[i, j] = 0  # из M
            else:
                Ix[i, j] = extend
                trace_Ix[i, j] = 1  # из Ix
            
            # Состояние Iy[i,j]
            open_gap = M[i-1, j] - gap_open
            extend = Iy[i-1, j] - gap_extend
            if open_gap >= extend:
                Iy[i, j] = open_gap
                trace_Iy[i, j] = 0  # из M
            else:
                Iy[i, j] = extend
                trace_Iy[i, j] = 2  # из Iy
    
    # Выбираем лучшее финальное состояние
    final_states = [M[n, m], Ix[n, m], Iy[n, m]]
    best_score = max(final_states)
    best_state = final_states.index(best_score)  # 0=M, 1=Ix, 2=Iy
    
    # Обратный ход
    seq1_align, seq2_align = '', ''
    i, j = n, m
    state = best_state
    
    while i > 0 or j > 0:
        if state == 0:  # M
            seq1_align = seq1[i-1] + seq1_align
            seq2_align = seq2[j-1] + seq2_align
            prev_state = trace_M[i, j]
            i -= 1
            j -= 1
            state = prev_state
        elif state == 1:  # Ix
            seq1_align = '-' + seq1_align
            seq2_align = seq2[j-1] + seq2_align
            prev_state = trace_Ix[i, j]
            j -= 1
            state = prev_state
        else:  # Iy
            seq1_align = seq1[i-1] + seq1_align
            seq2_align = '-' + seq2_align
            prev_state = trace_Iy[i, j]
            i -= 1
            state = prev_state
    
    return seq1_align, seq2_align, best_score, M, Ix, Iy

In [650]:
seq1_align, seq2_align, best_sc, best_matrix = Niddleman_Wunsch('ATGCAGCAGCAGCCA', 'ATATAT', [3, -3, -4])

In [652]:
af_seq1_align, af_seq2_align, af_best_sc, M, Ix, Iy = Needleman_Wunsch_Affine('ATGCAGCAGCAGCCA', 'ATATAT', [3, -3, 10, 1])

In [654]:
print(f"Выравнивание не аффинное:     Выравнивание аффинное:")
print(f"          {seq1_align}            {af_seq1_align}\n          {seq2_align}            {af_seq2_align}")
print(f"Score:             {best_sc}                       {af_best_sc}")

Выравнивание не аффинное:     Выравнивание аффинное:
          ATGCAGCAGCAGCCA            ATGCAGCAGCAGCCA
          AT-----A-TA---T            ATATA---------T
Score:             -30.0                       -18.0


**Вывод:** аффинное выравнивание лучше моделирует вставки/делеции, которые происходят достаточно часто в реальной жизни. Обычно они идут но несколько нуклеотидов, а не по одному.